# MANI: evaluating the AI CFO agents on OpenShift AI

**Owner:** Pair 1. Runs in an OpenShift AI workbench (project `cfo-llm`, no GPU needed).

Our rule: **code counts, the AI explains.** The rules engine computes every number; the model
(Llama 3.1 8B, vLLM on OpenShift AI) only picks which rule points each agent shows.
This notebook checks that promise against the running app and the running model:

1. **Routing**: does a question reach the right agent (Marek tax, Anna cash, Tomasz costs, Kasia currency)?
2. **Traceable numbers**: is every amount an agent says also present in the app's own data?
3. **Speed**: how long does an answer and a full agents' meeting take (demo budget: 24 s)?
4. **Model format**: how often does the model return valid JSON with only allowed point ids?

It only **reads**: `/api/ask` and `/api/boardroom` change no data. It never resets or approves anything.
Do not run it during a rehearsal or the demo: it uses the same GPU model as the live app.

In [ ]:
import json, re, time, statistics
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Inside the cluster the services are reachable directly. Default: the TEST copy of the app,
# so the live app is never disturbed. Switch to the live app only on purpose.
APP_URL = "http://warsztat-cfo-test.my-first-model.svc.cluster.local:8080"
# APP_URL = "http://warsztat-cfo.my-first-model.svc.cluster.local:8080"   # live app
MODEL_URL = "http://llama-31-8b-instruct-predictor.cfo-llm.svc.cluster.local:8080/v1"
MODEL_NAME = "llama-31-8b-instruct"
RUN_MEETINGS = 2        # full agents' meetings to time (each ~13 s on the 8B)
JSON_PROBES = 10        # direct model calls for the JSON-format check
TIMEOUT = 60

def get(path):
    r = requests.get(APP_URL + path, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

def post(path, body):
    t0 = time.perf_counter()
    r = requests.post(APP_URL + path, json=body, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json(), (time.perf_counter() - t0) * 1000

## 1. Is everything up?

In [ ]:
health = get("/api/llm/health")
models = requests.get(MODEL_URL + "/models", timeout=TIMEOUT).json()
print("App health :", health)
print("Model served:", [m["id"] for m in models["data"]], "| max context:", models["data"][0].get("max_model_len"))
assert health["mode"] == "openai_compat" and health["ok"], "The app is not using the real model: stop here."

## 2. Ground truth: every number the app's own data contains
We read the same read-only endpoints the UI uses and collect every number in them,
including numbers written inside rule texts (the `why` lines of the agents' cards).

In [ ]:
AMOUNT = re.compile(r"(-?\d{1,3}(?:[   ]\d{3})*(?:,\d{1,2})?|-?\d+(?:[.,]\d{1,2})?)\s*(zł|EUR|USD)")

def to_float(s):
    return round(float(re.sub(r"[   ]", "", s).replace(",", ".")), 2)

def amounts_in(text):
    return {(to_float(n), cur) for n, cur in AMOUNT.findall(text)}

known_numbers = set()
def walk(x):
    if isinstance(x, bool):
        return
    if isinstance(x, (int, float)):
        known_numbers.add(round(abs(float(x)), 2))
    elif isinstance(x, str):
        for n, _ in AMOUNT.findall(x):
            known_numbers.add(abs(to_float(n)))
        for n in re.findall(r"\d+(?:[.,]\d+)?", x.replace(" ", "")):
            known_numbers.add(abs(to_float(n)))
    elif isinstance(x, dict):
        for v in x.values():
            walk(v)
    elif isinstance(x, list):
        for v in x:
            walk(v)

for path in ["/api/state", "/api/forecast?days=90", "/api/expenses?days=30", "/api/alerts",
             "/api/actions", "/api/ksef/invoices", "/api/settings"]:
    walk(get(path))
print(f"{len(known_numbers)} distinct numbers found in the app's own data")

## 3. Ask every agent: routing, speed and traceable numbers

In [ ]:
QUESTIONS = {
    "marek":  ["How much VAT do I pay this month?", "When is my ZUS due?", "How much is my PIT advance?",
               "Why was a KSeF invoice rejected?", "What tax deadline is next?"],
    "anna":   ["Will I run out of money?", "Do I need a loan?", "Who still owes me money?",
               "What is my lowest balance this month?", "Is my cash safe in December?"],
    "tomasz": ["Which jobs were billed too low?", "Where do I lose margin?", "What are my biggest costs?",
               "Is my parts markup high enough?", "How can I earn more per car?"],
    "kasia":  ["What happens if the euro goes up?", "Should I lock the EUR rate?", "How much do I owe in euro?",
               "What is my currency risk?", "Is the zloty getting weaker?"],
}

rows = []
for expected, qs in QUESTIONS.items():
    for q in qs:
        for named in (True, False):   # with the agent chosen, and picked by topic (routing)
            body = {"question": q, **({"cfo": expected} if named else {})}
            res, ms = post("/api/ask", body)
            said = amounts_in(res["text"])
            untraced = sorted(f"{a:,.2f} {c}" for a, c in said if abs(a) not in known_numbers)
            rows.append({"agent": expected, "question": q, "named": named, "answered_by": res["cfo"],
                         "routed_ok": res["cfo"] == expected, "ms": round(ms), "amounts": len(said),
                         "untraced": ", ".join(untraced), "answer": res["text"]})
df = pd.DataFrame(rows)
df["untraced_n"] = df.untraced.map(lambda s: len(s.split(", ")) if s else 0)
print(f"{len(df)} answers collected")

In [ ]:
by_topic = df[~df.named]
summary = pd.DataFrame({
    "answers": df.groupby("agent").size(),
    "routing ok (picked by topic)": by_topic.groupby("agent").routed_ok.mean().map("{:.0%}".format),
    "median ms": df.groupby("agent").ms.median().astype(int),
    "amounts said": df.groupby("agent").amounts.sum(),
    "amounts not in app data": df.groupby("agent").untraced_n.sum(),
})
print("Routing by topic overall:", f"{by_topic.routed_ok.mean():.0%}")
total_amounts = int(df.amounts.sum())
untraced_total = int(summary["amounts not in app data"].sum())
print(f"Amounts traceable to the app's data: {total_amounts - untraced_total}/{total_amounts}")
summary

Amounts "not in app data" are listed below for a human to check. They can be honest rule results that
are not stored as one field (for example *balance minus buffer*), or an invented number. The model never
writes text in our app, so any untraced amount points at a rule, not at the model.

In [ ]:
df[df.untraced != ""][["agent", "question", "untraced", "answer"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby("agent").ms.median().plot.bar(ax=ax[0], color="#0f5132", title="Median answer time per agent (ms)")
by_topic.groupby("agent").routed_ok.mean().mul(100).plot.bar(ax=ax[1], color="#b45309",
    title="Questions routed to the right agent (%)", ylim=(0, 105))
for a in ax: a.set_xlabel("")
plt.tight_layout(); plt.show()

## 4. A full agents' meeting: does it fit the 24 s demo budget?

In [ ]:
meetings = []
for i in range(RUN_MEETINGS):
    res, ms = post("/api/boardroom", {"topic": "Will we survive December?"})
    meetings.append({"run": i + 1, "seconds": round(ms / 1000, 1), "turns": len(res["turns"]),
                     "decisions": len(res["decisionActionIds"])})
pd.DataFrame(meetings)

## 5. The model itself: valid JSON with allowed ids only
This is the model's whole job in our app: pick point ids from a list and answer in JSON.
We ask it directly (no app in between) and count how often the reply is usable.

In [ ]:
POINTS = {"p1": "VAT 14 230,00 zł due on 25 November", "p2": "Balance drops to -20 275,00 zł on 20 December",
          "p3": "3 jobs billed below the labour rate", "p4": "1500 EUR to pay, rate rising"}
prompt = ("You help a car repair shop owner. From these points pick the 1-3 most important for the question "
          "'Will I have money in December?'. Reply ONLY with JSON like {\"points\": [\"p2\"]}.\n"
          + "\n".join(f"{k}: {v}" for k, v in POINTS.items()))
probes = []
for i in range(JSON_PROBES):
    t0 = time.perf_counter()
    r = requests.post(MODEL_URL + "/chat/completions", timeout=TIMEOUT, json={
        "model": MODEL_NAME, "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 60, "temperature": 0.4}).json()
    ms = (time.perf_counter() - t0) * 1000
    text = r["choices"][0]["message"]["content"]
    try:
        picked = json.loads(text[text.index("{"): text.rindex("}") + 1]).get("points", [])
        ok = bool(picked) and all(p in POINTS for p in picked)
    except Exception:
        picked, ok = [], False
    toks = r.get("usage", {}).get("completion_tokens", 0)
    probes.append({"ok": ok, "picked": picked, "ms": round(ms), "tokens/s": round(toks / (ms / 1000), 1) if toks else None})
probes = pd.DataFrame(probes)
print(f"Usable JSON: {probes.ok.sum()}/{len(probes)} | median {probes.ms.median():.0f} ms | "
      f"~{probes['tokens/s'].median()} tokens/s")
probes

## Result in one line (paste into the Pair 1 log)

In [ ]:
line = (f"Eval on {MODEL_NAME}: routing by topic {by_topic.routed_ok.mean():.0%}, "
        f"amounts traceable {total_amounts - untraced_total}/{total_amounts}, "
        f"median answer {df.ms.median():.0f} ms, meetings {', '.join(str(m['seconds']) for m in meetings)} s, "
        f"model JSON usable {probes.ok.sum()}/{len(probes)}")
print(line)